# Training the reviewer on Google Colab
Runtime -> Change runtime type -> **T4 GPU**. Run the cells in order. Upload the project as a zip (or clone it from your GitHub) first.

In [ ]:
!unzip -q intelligent-code-reviewer.zip -d /content 2>/dev/null; %cd /content/intelligent-code-reviewer
!pip install -q -r requirements-train.txt

## Step A - Get the data
Download the CodeReviewer dataset (Zenodo, link in the paper by Li et al., 2022) and the CVEfixes database (Zenodo, Bhandari et al., 2021). Put them in `data/raw/`.

In [ ]:
!python training/make_bug_samples.py cvefixes --db data/raw/CVEfixes.db --out data/raw/cvefixes.jsonl --languages Python Java

In [ ]:
!python training/prepare_data.py --codereviewer data/raw/msg-train.jsonl --bugs data/raw/cvefixes.jsonl --out data/processed --max-review 20000
!cat data/processed/data_stats.json

## Step B - LoRA fine-tuning

In [ ]:
!python training/train_lora.py --train data/processed/train.jsonl --val data/processed/val.jsonl \
   --model Qwen/Qwen2.5-Coder-1.5B-Instruct --out outputs/lora-reviewer --load-4bit --epochs 2

## Step C - Tune the false-positive filter on the validation set

In [ ]:
!python training/tune_threshold.py --val data/processed/val_bugs.jsonl --backend hf \
   --model Qwen/Qwen2.5-Coder-1.5B-Instruct --adapter outputs/lora-reviewer --load-4bit --out outputs

## Step D - Evaluate (full system, ablations, baseline 1), then baseline 2 with the base model

In [ ]:
!python evaluation/run_eval.py --test data/processed/test_bugs.jsonl --comments data/processed/test_comments.jsonl \
   --backend hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct --adapter outputs/lora-reviewer --load-4bit \
   --filter-config outputs/filter_config.json --out results
!python evaluation/run_eval.py --test data/processed/test_bugs.jsonl --variants llm_diff_only \
   --backend hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct --load-4bit --out results
!cat results/metrics.md

In [ ]:
# Save everything to Google Drive so a disconnect does not lose your work
from google.colab import drive; drive.mount('/content/drive')
!cp -r outputs results data/processed/data_stats.json /content/drive/MyDrive/